# 12 — Preserve annotated objects and publish by storage category

For export-only rows, copy the original object byte-for-byte, including corrected X, raw layers, UMAPs and annotations; do not run QC/segmentation/integration.
For other rows, publish the completed new SpatialData products. Object, plot and data destinations are independent CSV columns.
S3 staging/publication require explicit policy opt-ins and valid IAM/bucket permissions. `_SUCCESS.json` is written last. Never treat a partial prefix as complete; no automatic overwrite, deletion or scratch cleanup.
Finish notebook 11 before publishing a run that should contain scVI annotations.

In [ ]:
from pathlib import Path
import os, sys, json
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "vhd" / "control").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter in the extracted pipeline folder (or its notebooks folder).")
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from vhd.control.manifest import load_project, check_policy, save_plan, sample_layout
MANIFEST = Path(os.environ.get("VHD_MANIFEST", ROOT / "config" / "samples.csv"))
SETTINGS = Path(os.environ.get("VHD_SETTINGS", ROOT / "config" / "settings.json"))
if not MANIFEST.exists() or not SETTINGS.exists():
    raise FileNotFoundError("Copy a supplied samples.*.csv to config/samples.csv and settings.g5_24xlarge.example.json to config/settings.json; edit paths and policy first.")
PROJECT = load_project(MANIFEST, SETTINGS)
check_policy(PROJECT)
# Default is a dry run. Set True here only after reviewing the printed plan.
EXECUTE = os.environ.get("VHD_EXECUTE", "0") == "1"
# Optional pilot selection, e.g. ["StudyLegacy__Sample01"]. None selects all applicable rows.
SAMPLE_KEYS = None


## Build export-only products (other stages skipped)

In [ ]:
from vhd.compute.launch import launch_samples
launch_samples(PROJECT, ["export_existing"], execute=EXECUTE, sample_keys=SAMPLE_KEYS)

## Preview destinations / publish completed per-sample products

In [ ]:
from vhd.control.runner import publish_sample, publish_integration
rows = [r for r in PROJECT["rows"] if SAMPLE_KEYS is None or r["sample_key"] in SAMPLE_KEYS]
for row in rows:
    print(row["sample_key"])
    print(publish_sample(PROJECT, row, execute=EXECUTE))

## Optional completed integration product publication

In [ ]:
PUBLISH_INTEGRATION = False
if PUBLISH_INTEGRATION:
    groups = sorted({r["integration_group"] for r in PROJECT["rows"] if r["integration_group"]})
    for group in groups:
        print(publish_integration(PROJECT, group, execute=EXECUTE))